In [0]:
%run ../0-common/env-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"

silver_table = f"{catalog_name}.{silver_schema}.results"

In [0]:
from pyspark.sql import functions as F

In [0]:
results_df = spark.read.table(bronze_table)

In [0]:
results_selected_df = results_df.drop("url")

In [0]:
results_renamed_df = (
    results_selected_df
    .withColumnsRenamed({
        "raceName": "race_name",
        "constructorId": "constructor_id",
        "driverId": "driver_id",
        "positionText": "position_text",
        "date": "race_date",
        "grid": "grid_position",
        "number": "car_number",
        "position": "final_position",
        "positionText": "final_position_text"
    })  
)

In [0]:
results_distinct_df = (
    results_renamed_df
    .filter(
        F.col("round").isNotNull() &
        F.col("season").isNotNull() &
        F.col("constructor_id").isNotNull() &
        F.col("driver_id").isNotNull()
    )
    .dropDuplicates(["round", "season", "constructor_id", "driver_id"])
)

In [0]:
results_final_df = (
    results_distinct_df
    .withColumn("race_name", F.initcap("race_name"))
)

In [0]:
(
    results_final_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)